# 13. Lecke - Ügynök memória


## Beállítás

Ez a jegyzetfüzet bemutatja, hogyan lehet utazási foglaló ügynököt építeni **perzisztens memóriával** a **Microsoft Agent Framework** (MAF) használatával.

Megtanulhatod, hogy az ügynök különböző memóriatípusai — munkamemória, rövid távú és hosszú távú memória — hogyan befolyásolják az információ megőrzését és felhasználását a beszélgetések során.

**Előfeltételek:**
- Egy Azure AI Foundry projekt egy telepített chat modellel (pl. `gpt-4o-mini`).
- Bejelentkezve az Azure CLI-be — futtasd a `az login` parancsot a terminálodban.
- `AZURE_AI_PROJECT_ENDPOINT` — az Azure AI Foundry projekt végpontja.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — a telepített modell neve.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Az ügynök memória típusai

A MI ügynökök különböző memóriafajtákat használnak, melyek mindegyike eltérő célt szolgál:

### Munkamemória
Magát a beszélgetési szálat jelenti — az egyetlen munkamenet során váltott üzeneteket. Az ügynök visszautalhat a korábbi üzenetekre ugyanebben a szálban a koherencia fenntartása érdekében. A MAF-ban ez a **`agent.create_session()`** segítségével jön létre, amely `AgentSession`-t ad vissza.

### Rövid távú memória
Olyan információk, amelyek egy feladat vagy munkamenet ideje alatt fennmaradnak, de nem tárolódnak tartósan. Például egy többlépéses tervező beszélgetés során az ügynök tényszerű adatokat gyűjthet, majd ezek felhasználásával elkészítheti a végső útitervet.

### Hosszú távú memória
Olyan preferenciák és tények, amelyek **munkamenetek között** is fennmaradnak. Egy visszatérő felhasználónak nem kell megismételnie az étrendi megszorításait vagy utazási stílusát. A hosszú távú memóriát rendszerint külső tároló — adatbázis, állomány vagy vektorindex — támogatja, és eszközökön keresztül érhető el az ügynök számára.


## Munkamemória munkamenetekkel

A memória legegyszerűbb formája a beszélgetési munkamenet. Ha ugyanazt a munkamenetet (az `agent.create_session()` segítségével létrehozott) adod át egymás utáni `agent.run()` hívásoknál, az ügynök látja az adott beszélgetés teljes előzményeit, és fel tudja idézni a korábbi részleteket.

Hozzunk létre egy utazási ügynököt, és demonstráljuk a munkamemóriát.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Az ügynök helyesen idézte fel a költségvetést, mert mindkét üzenet ugyanahhoz a munkamenethez tartozik. Ez a **munkamemória** — csak a munkamenet élettartamáig létezik.

### Mi történik egy új szállal?

Ha **új** munkamenetet hozunk létre, az ügynök nem emlékszik az előző beszélgetésre:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Hosszú távú memória minta

Ahhoz, hogy a felhasználói preferenciákat **munkamenetek között** meg tudjuk jegyezni, szükségünk van egy olyan tartós tárolóra, amely a beszélgetési szálon kívül él. Az ügynök ehhez a tárolóhoz **eszközökön** keresztül fér hozzá — olyan függvényeken, amelyeket információ mentésére és lekérésére hívhat meg.

Alább egy egyszerű memóriabeli preferenciatárolót valósítunk meg (éles környezetben ezt adatbázissal vagy vektorindexszel támasztanánk alá), és eszközökként tesszük elérhetővé az ügynök számára.

### Architektúra
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### 2. forgatókönyv — Sarah hetek múlva visszatér

Sarah egy **teljesen új szálat** indít (új munkamenetet szimulálva). A munkamemória üres, de a hosszú távú preferenciák tárolója még mindig tartalmazza az ő adatait. Az ügynöknek elő kell hívnia ezeket, és fel kell használnia a személyre szabott ajánlásokhoz.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Összefoglaló

Ebben a leckében háromféle ügynök memóriát és azok megvalósítását tanultad meg a Microsoft Agent Framework segítségével:

| Memória típusa | MAF mechanizmus | Élettartam |
|---|---|---|
| **Munkamemória** | `agent.create_session()` | Egyetlen beszélgetés |
| **Rövid távú** | Felhalmozott kontextus egy szálon belül | Egyetlen feladat / munkamenet |
| **Hosszú távú** | Külső tároló, amit `@tool` függvényeken keresztül ér el | Munkamenetek között |

### Főbb tanulságok
1. **`agent.create_session()`** biztosítja a munkamemóriát — az ügynök látja a teljes beszélgetés történetét egy munkameneten belül.
2. **Az új munkamenetek elveszítik a kontextust** — hosszú távú memória nélkül az ügynök nem emlékszik korábbi beszélgetésekre.
3. **`@tool` függvények** hidat képeznek — lehetővé teszik az ügynök számára az információk elmentését és előhívását egy tartós tárolóból.
4. **A személyre szabás idővel javul** — minél több preferencia kerül eltárolásra, annál jobb az ügynök ajánlása.

### Valós alkalmazások
- **Ügyfélszolgálat**: ügyfél előzmények és preferenciák megjegyzése
- **Személyi asszisztensek**: kontextus megtartása napokon vagy heteken át
- **Egészségügy**: betegek adatainak és preferenciáinak nyomon követése
- **E-kereskedelem**: személyre szabott vásárlás előzmények alapján

### Következő lépések
- Memória helyettesítése adatbázissal vagy vektortárral (pl. Azure AI Search)
- Memória lejárati idő bevezetése időérzékeny információkhoz
- Több ügynökös rendszerek építése megosztott memóriával
- Felfedezni a Cognee jegyzetfüzetet tudásgráf-alapú memóriához


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Nyilatkozat**:  
Ezt a dokumentumot az AI fordító szolgáltatás [Co-op Translator](https://github.com/Azure/co-op-translator) segítségével fordítottuk. Bár a pontosságra törekszünk, kérjük vegye figyelembe, hogy az automatikus fordítások hibákat vagy pontatlanságokat tartalmazhatnak. Az eredeti, anyanyelvi dokumentum tekintendő hiteles forrásnak. Fontos információk esetén professzionális, emberi fordítást javaslunk. Nem vállalunk felelősséget az esetleges félreértésekért vagy helytelen értelmezésekért, amelyek a fordítás használatából erednek.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
